# <font color='black'> Session-05-Class Example: Greenwashing <font>


## <font color='blue'> 1. Load and Explore the Dataset (json) <font>
    


In [25]:
#!python3 --version
#!pip install scikit-learn
#!python3 -m pip install scikit-learn


import pandas as pd
import json


# Load the dataset
path = "company_dataset.json"
mydata = pd.read_json(path)

# Display the first few rows of the dataset to understand its structure
#print(mydata.dtypes)

print(mydata['label'].value_counts())
mydata.head()


label
no_greenwashing    4596
greenwashing        404
Name: count, dtype: int64


,comment,label
0,ine . </s> Home My Network </s> WA RERAU P UR ...,no_greenwashing
1,"1/13/22, 1:23 PM Cummins Positions on Key Issu...",no_greenwashing
2,"24/01/2023, 15:07 Italy can boost African gas ...",greenwashing
3,"15/02/2023, 15:40 (21) EDF EU Affairs on Twitt...",no_greenwashing
4,"06/06/2022, 15:17 Post | Feed | LinkedIn </s> ...",no_greenwashing


## <font color='blue'> 2. Data Cleaning and preprocessing <font>

In [26]:
import re
import string

# no need to lowercase or remove stoopwords

# Function to clean text
def data_pre_process(text):
    text = re.sub(f"[{string.punctuation}]", "", text)  # Remove punctuation
    text = re.sub("\d+", "", text)  # Remove numbers
    return text

# Apply text cleaning to all comments
mydata['comment'] = mydata['comment'].apply(data_pre_process)
mydata.head()

,comment,label
0,ine s Home My Network s WA RERAU P UR Y Renae...,no_greenwashing
1,PM Cummins Positions on Key Issues Cummins ...,no_greenwashing
2,Italy can boost African gas imports Eni CEO ...,greenwashing
3,EDF EU Affairs on Twitter EU ETS carbon pri...,no_greenwashing
4,Post Feed LinkedIn s httpswwwlinkedincomfe...,no_greenwashing


## <font color='blue'> 3. Split Data into Training and Testing Sets <font>


In [27]:

from sklearn.model_selection import train_test_split

# Define X (features) and y (labels)
X = mydata['comment']
y = mydata['label']

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)
print(f"Training samples: {len(X_train)}, Testing samples: {len(X_test)}")


Training samples: 4000, Testing samples: 1000


## <font color='blue'> 4. Convert text to numerical format using TF-IDF
<font>

In [28]:

from sklearn.feature_extraction.text import TfidfVectorizer

#converts raw text into numerical TF-IDF vectors 
# Link: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html#
vectorizer = TfidfVectorizer()

#.fit_transform(X_train) learns the vocabulary and computes TF-IDF scores
X_train_tfidf = vectorizer.fit_transform(X_train)

#.transform(X_test) applies the learned transformation to test data
X_test_tfidf = vectorizer.transform(X_test)

#print (no. of documents (each row), no. of unique words (features))
print("TF-IDF Feature Sample:")
print(X_train_tfidf.shape)


TF-IDF Feature Sample:
(4000, 43095)


## <font color='blue'> 5. Train and Test the Logistic Regression Model <font>

In [29]:
from sklearn.linear_model import LogisticRegression

# Creating the model
lrm = LogisticRegression(max_iter=1000)

# Training the model---finding the best coefficients (weights) for each feature
# so the model can predict the correct class.
lrm.fit(X_train_tfidf, y_train)

# Making predictions to test the model---apply the learned coefficients from training
predicted = lrm.predict(X_test_tfidf)

## <font color='blue'> 6. Model Evaluation <font>

In [30]:
from sklearn.metrics import classification_report, confusion_matrix

print ("---Confusion_matrix---")
print(confusion_matrix(y_test, predicted))
print("\n")


print("\t \t ----Model Performance----")
print(classification_report(y_test, predicted))

---Confusion_matrix---
[[  9  89]
 [  1 901]]


	 	 ----Model Performance----
                 precision    recall  f1-score   support

   greenwashing       0.90      0.09      0.17        98
no_greenwashing       0.91      1.00      0.95       902

       accuracy                           0.91      1000
      macro avg       0.91      0.55      0.56      1000
   weighted avg       0.91      0.91      0.88      1000



## <font color='blue'>  Model Evaluation Explanation <font>

###  Accuracy: How often is the model correct?
The model makes **correct predictions 91% of the time**, meaning that out of **100 samples**, it correctly classifies **91**.

---

###  Precision: How often is the model right when it predicts a class?
- When the model **predicts "greenwashing" (True)**, it is **correct 90%** of the time.
- When the model **predicts "no greenwashing" (False)**, it is **correct 91%** of the time.

---

### Recall: How good is the model at catching actual cases?
- Out of **100 actual greenwashing cases**, the model **only identifies 9** correctly (**low recall**).
- Out of **100 actual no greenwashing cases**, the model **identifies all 100** (**high recall**).

---

### F1-Score: A balance between Precision and Recall
- **Greenwashing (True)**: **F1-score = 0.17**, which is **very low** due to poor recall.
- **No Greenwashing (False)**: **F1-score = 0.95**, showing high confidence in this class.

---

### Support: Number of actual cases in the dataset
- **98 cases** of Greenwashing (True)
- **902 cases** of No Greenwashing (False)


## <font color='blue'> Overall Model Performance<font> 
    
### Macro Average
- **Treats both classes equally**, no matter how many times they appear.
- **Precision: 0.91 | Recall: 0.55 | F1-Score: 0.56**
- The **low recall for greenwashing** reduces the overall score.

### Weighted Average
- **Gives more weight to "no greenwashing" because it appears more often**.
- **Precision: 0.91 | Recall: 0.91 | F1-Score: 0.88**
- **Looks good overall, but hides poor performance on greenwashing cases**.

###  Weighted Precision Calculation

Weighted Precision ={(0.90 * 98) + (0.91 * 902)}/{1000} = 0.91
